# Cafetería: análisis estadístico de llegadas y tiempos de preparación

Este notebook resuelve cuatro preguntas usando Python y librerías estadísticas: distribución de llegadas, distribución del tiempo de preparación, regla 68-95-99.7 y Teorema del Límite Central.

In [ ]:
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt

# Parámetros proporcionados por el problema
lambda_llegadas = 12       # clientes por hora
media_preparacion = 4.2   # minutos
desv_preparacion = 0.8     # minutos
n = 40                     # tamaño de muestra


## 1. ¿Qué distribución modela las llegadas por hora?

Usamos una **distribución de Poisson** porque estamos contando cuántos clientes llegan durante un intervalo fijo de tiempo (una hora), y el promedio de llegadas es 12 por hora.

Por tanto:

$$X \sim Poisson(\lambda=12)$$

La Poisson es apropiada para conteos de eventos en un intervalo cuando se supone una tasa media aproximadamente constante e independencia entre llegadas.

In [ ]:
x = np.arange(0, 31)
probabilidades = stats.poisson.pmf(x, mu=lambda_llegadas)

plt.figure(figsize=(9, 5))
plt.bar(x, probabilidades)
plt.xlabel('Clientes que llegan en una hora')
plt.ylabel('Probabilidad')
plt.title('Distribución de Poisson: llegadas por hora')
plt.show()

# Ejemplo: probabilidad de que lleguen exactamente 12 clientes
p_12 = stats.poisson.pmf(12, mu=lambda_llegadas)
print(f'P(X = 12) = {p_12:.4f} ({p_12*100:.2f}%)')


## 2. ¿Qué distribución usar para el tiempo de preparación?

Usamos una **distribución normal** como modelo aproximado, porque se proporcionan una media y una desviación estándar y el tiempo de preparación se plantea alrededor de un valor central.

$$X \sim N(\mu=4.2,\sigma=0.8)$$

Esto supone que los tiempos se distribuyen aproximadamente de forma simétrica alrededor de 4.2 minutos.

In [ ]:
x_tiempo = np.linspace(1, 7, 500)
densidad = stats.norm.pdf(x_tiempo, loc=media_preparacion, scale=desv_preparacion)

plt.figure(figsize=(9, 5))
plt.plot(x_tiempo, densidad)
plt.axvline(media_preparacion, linestyle='--', label='Media = 4.2 min')
plt.axvline(3.4, linestyle=':', label='3.4 min')
plt.axvline(5.0, linestyle=':', label='5.0 min')
plt.xlabel('Tiempo de preparación (minutos)')
plt.ylabel('Densidad')
plt.title('Distribución Normal del tiempo de preparación')
plt.legend()
plt.show()

# 3.4 y 5.0 están exactamente a +/- 1 desviación estándar de la media.
p_34_50 = stats.norm.cdf(5.0, media_preparacion, desv_preparacion) - stats.norm.cdf(3.4, media_preparacion, desv_preparacion)
print(f'P(3.4 <= X <= 5.0) = {p_34_50:.4f} ({p_34_50*100:.2f}%)')


## 3. Regla 68-95-99.7

Para una distribución normal:

- Aproximadamente **68%** de los datos están entre $\mu-\sigma$ y $\mu+\sigma$.
- Aproximadamente **95%** están entre $\mu-2\sigma$ y $\mu+2\sigma$.
- Aproximadamente **99.7%** están entre $\mu-3\sigma$ y $\mu+3\sigma$.

En este problema, $\mu=4.2$ y $\sigma=0.8$.

In [ ]:
for k in [1, 2, 3]:
    inferior = media_preparacion - k * desv_preparacion
    superior = media_preparacion + k * desv_preparacion
    porcentaje = (stats.norm.cdf(superior, media_preparacion, desv_preparacion)
                  - stats.norm.cdf(inferior, media_preparacion, desv_preparacion)) * 100
    print(f'{k}σ: entre {inferior:.1f} y {superior:.1f} min -> {porcentaje:.2f}%')


### Interpretación

Como 3.4 = 4.2 - 0.8 y 5.0 = 4.2 + 0.8, el intervalo de 3.4 a 5.0 minutos corresponde a **±1 desviación estándar**.

Por la regla 68-95-99.7, aproximadamente **68% de los cafés** se prepararían entre 3.4 y 5.0 minutos.

El cálculo exacto de la normal estándar da aproximadamente **68.27%**.

## 4. ¿Por qué una muestra de 40 clientes puede dar un promedio confiable?

Aquí usamos el **Teorema del Límite Central (TLC)**. Si tomamos muestras aleatorias independientes de tamaño suficientemente grande, la distribución de la media muestral se aproxima a una distribución normal, incluso si la distribución original de satisfacción no es perfectamente normal.

Con $n=40$, se considera una muestra suficientemente grande para aplicar habitualmente el TLC, bajo condiciones razonables de independencia y ausencia de una distribución extremadamente problemática.

Si la población de satisfacción tiene media $\mu$ y desviación estándar $\sigma$, entonces:

$$\bar X \approx N\left(\mu,\frac{\sigma}{\sqrt{40}}\right)$$

La desviación estándar de la media muestral es el **error estándar**:

$$SE=\frac{\sigma}{\sqrt{40}}$$

Por lo tanto, el promedio de 40 clientes es más estable que la respuesta de un solo cliente y permite estimar mejor la satisfacción promedio de la población.

In [ ]:
print(f'√40 = {np.sqrt(n):.4f}')
print('Error estándar = σ / √40')
print('Si, por ejemplo, σ = 10 puntos de satisfacción:')
print(f'SE = 10 / √40 = {10/np.sqrt(n):.4f} puntos')

# Simulación sencilla del TLC: población normal con media 80 y desviación 10.
rng = np.random.default_rng(42)
medias = [rng.normal(80, 10, n).mean() for _ in range(5000)]

plt.figure(figsize=(9, 5))
plt.hist(medias, bins=40, density=True)
plt.xlabel('Media de satisfacción de una muestra de 40 clientes')
plt.ylabel('Densidad')
plt.title('Ilustración del Teorema del Límite Central')
plt.show()


## Conclusiones

1. **Llegadas:** se modelan con **Poisson(12)** porque se cuentan clientes que llegan durante una hora y la tasa promedio es de 12 clientes/hora.
2. **Preparación:** se puede modelar aproximadamente con **Normal(4.2, 0.8)**.
3. **Entre 3.4 y 5.0 minutos:** representan $\mu\pm\sigma$, por lo que aproximadamente **68.27%** de los cafés caen en ese intervalo bajo el modelo normal.
4. **Muestra de 40 clientes:** el TLC hace que la media muestral tenga una distribución aproximadamente normal y un error estándar menor que la variabilidad individual, por lo que el promedio de la muestra es una estimación más estable de la satisfacción poblacional.

### Nota importante
La confiabilidad exacta del promedio de satisfacción depende también de la variabilidad de las respuestas y de cómo se seleccionaron los 40 clientes. El tamaño 40 ayuda a aplicar el TLC, pero no garantiza por sí solo que la muestra sea representativa.